In [1]:
import os
import json
import chromadb
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

In [2]:
load_dotenv()

True

In [3]:

API_KEY = os.getenv('OPENAI_API_KEY')
embedding_model = OpenAIEmbeddings(model='text-embedding-3-large', api_key=API_KEY)

In [4]:
CHROMA_API_KEY = os.getenv('CHROMA_API_KEY')
CHROMA_TENANT = os.getenv('CHROMA_TENANT')
CHROMA_DATABASE = os.getenv('CHROMA_DATABASE')

In [5]:
client = chromadb.CloudClient(
  api_key=CHROMA_API_KEY,
  tenant=CHROMA_TENANT,
  database=CHROMA_DATABASE
)

collection = client.get_or_create_collection("organization_data")

In [6]:
old_path = "data/old_internal_data.jsonl"
new_path = "data/internal_data.jsonl"
print(os.path.exists(old_path), os.path.exists(new_path))

old_data = {}
if os.path.exists(old_path):
    with open(old_path, "r") as f:
        for line in f:
            entry = json.loads(line)
            old_data[entry["docId"]] = entry.get("updated_at")

changed_entries = []
with open(new_path, "r") as f:
    for i, line in enumerate(f):
        entry = json.loads(line)
        doc_id = entry.get("docId")
        updated_at = entry.get("updated_at")

        if doc_id not in old_data or old_data[doc_id] != updated_at:
            changed_entries.append((i, entry))

print(f"Found {len(changed_entries)} new/updated entries.")

True True
Found 1 new/updated entries.


In [7]:
new_chunks = []
global_ids = []
global_metadata = []

for i, entry in changed_entries:
    combined_text = ""
    for key, value in reversed(list(entry.items())):
        if key not in ['tags', 'chunk', 'docId']:
            combined_text += f"{key}: {value} "
    combined_text = combined_text.strip()

    new_chunks.append(combined_text)
    global_ids.append(f"{entry.get('docId')}_{entry.get('chunk', i)}")
    global_metadata.append({
        "docId": entry.get("docId"),
        "updated_at": entry.get("updated_at")
    })

In [8]:
batch_size = 5
embeddings_list = []
for i in range(0, len(new_chunks), batch_size):
    batch = new_chunks[i:i+batch_size]
    batch_embeddings = embedding_model.embed_documents(batch)
    embeddings_list.extend(batch_embeddings)

In [9]:
print(global_ids)

['17dcd5aea08c065d_1']


In [10]:
if len(embeddings_list) != len(new_chunks):
    raise ValueError("Embeddings list and JSONL entries count do not match!")

In [11]:
if embeddings_list:
    collection.upsert(
        ids=global_ids,
        documents=new_chunks,
        embeddings=embeddings_list,
        metadatas=global_metadata
    )
    print(f"Upserted {len(new_chunks)} changed entries into Chroma.")
else:
    print("No changes detected — nothing to update.")

Upserted 1 changed entries into Chroma.
